In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [15]:
# CBAM Attention Module
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, in_channels // reduction_ratio, kernel_size=1, padding=0)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(in_channels // reduction_ratio, in_channels, kernel_size=1, padding=0)
        self.sigmoid = nn.Sigmoid()

        # Spatial Attention
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        # Channel Attention
        max_pool_out = self.max_pool(x)
        avg_pool_out = self.avg_pool(x)
        channel_attn = self.fc2(self.relu(self.fc1(max_pool_out))) + self.fc2(self.relu(self.fc1(avg_pool_out)))
        channel_attn = self.sigmoid(channel_attn)
        x = x * channel_attn

        # Spatial Attention
        max_pool_out = torch.max(x, dim=1, keepdim=True)[0]
        avg_pool_out = torch.mean(x, dim=1, keepdim=True)
        spatial_attn = self.spatial_conv(torch.cat([max_pool_out, avg_pool_out], dim=1))
        spatial_attn = self.sigmoid(spatial_attn)
        x = x * spatial_attn

        return x

In [7]:
  def forward(self, x):
        # Channel Attention
        max_pool_out = self.max_pool(x)
        avg_pool_out = self.avg_pool(x)
        channel_attn = self.fc2(self.relu(self.fc1(max_pool_out))) + self.fc2(self.relu(self.fc1(avg_pool_out)))
        channel_attn = self.sigmoid(channel_attn)
        x = x * channel_attn

        # Spatial Attention
        max_pool_out = torch.max(x, dim=1, keepdim=True)[0]
        avg_pool_out = torch.mean(x, dim=1, keepdim=True)
        spatial_attn = self.spatial_conv(torch.cat([max_pool_out, avg_pool_out], dim=1))
        spatial_attn = self.sigmoid(spatial_attn)
        x = x * spatial_attn

        return x


In [8]:
# Depthwise Separable Convolution Block
class DSCBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DSCBlock, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.relu(x)


In [9]:
# Encoder Block with CBAM
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv1 = DSCBlock(in_channels, out_channels)
        self.conv2 = DSCBlock(out_channels, out_channels)
        self.cbam = CBAM(out_channels)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.cbam(x)
        p = self.pool(x)
        return x, p

In [10]:
# Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv1 = DSCBlock(in_channels, out_channels)
        self.conv2 = DSCBlock(out_channels, out_channels)

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

In [11]:
# U-Net with CBAM
class UNetCBAM(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetCBAM, self).__init__()
        self.enc1 = EncoderBlock(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        self.bottleneck = DSCBlock(512, 1024)

        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        s1, p1 = self.enc1(x)
        s2, p2 = self.enc2(p1)
        s3, p3 = self.enc3(p2)
        s4, p4 = self.enc4(p3)

        # Bottleneck
        b = self.bottleneck(p4)

        # Decoder
        d4 = self.dec4(b, s4)
        d3 = self.dec3(d4, s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)

        # Output
        return self.final_conv(d1)

In [16]:
# Testing the model with a sample input
if __name__ == '__main__':
    model = UNetCBAM(in_channels=12, out_channels=1)  # Adjust input channels for precipitation maps
    x = torch.randn(1, 12, 288, 288)  
    output = model(x)
    print(f"Output shape: {output.shape}")

Output shape: torch.Size([1, 1, 288, 288])
